# SolarCast Lite: one-hour-ahead solar power forecasting
Veritas AI High School Hackathon 2026, AI + Climate track.

**Problem and user.** A small solar plant (50 kW) feeds the grid, and whoever operates it needs to know how much power
it will produce in the next hour: to balance supply and demand, and to decide when to run flexible loads or storage.
Clouds make the next hour hard to guess. We forecast the power of the next hour from recent power, solar geometry and
weather, with scikit-learn, and compare against two simple baselines.

**Important.** Results are a *historical replay* on held-out test data. The weather inputs are Open-Meteo *reanalysis*
(what the weather actually was), not live forecasts. This project is a prototype.

## Setup (run this first)
This cell makes the notebook work both in Google Colab and on a laptop. In Colab it clones (or updates) the
GitHub repo into `/content/SolarCast-Lite`; locally it finds the repo root. Then it puts `src/` on the Python path
and turns on autoreload, so edits to `src/solarcast/*.py` are picked up without restarting.
If the repo is private, add a Colab Secret named `GITHUB_TOKEN` (see `docs/COLAB_GUIDE.md`). The token is never printed.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nashwu0117/SolarCast-Lite.git"
IN_COLAB = "google.colab" in sys.modules


def _git(args, cwd=None, secret=None, check=True):
    """Run git, hiding the token if it shows up in any output."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if secret:
        out = out.replace(secret, "***")
    if check and r.returncode != 0:
        raise RuntimeError("git " + args[0] + " failed:\n" + out)
    return r.returncode, out


if IN_COLAB:
    REPO = Path("/content/SolarCast-Lite")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None  # no secret: fine for a public repo
    auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    if not REPO.exists():
        _git(["clone", auth_url, str(REPO)], secret=token)
        _git(["remote", "set-url", "origin", REPO_URL], cwd=REPO)  # keep the token out of .git/config
    elif not (REPO / ".git").exists():
        print("Using the uploaded copy in", REPO, "(not a git clone, so no pull).")
    else:
        _git(["remote", "set-url", "origin", auth_url], cwd=REPO, secret=token)
        code, out = _git(["pull", "--ff-only"], cwd=REPO, secret=token, check=False)
        _git(["remote", "set-url", "origin", REPO_URL], cwd=REPO)
        if code != 0:
            print("git pull did not fast-forward (local edits?). Continuing with the current files.\n" + out)
    os.chdir(REPO)
    if importlib.util.find_spec("gradio") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
else:
    REPO = Path.cwd().resolve()
    while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
        REPO = REPO.parent
    os.chdir(REPO)

# Editable installs do not take effect in Colab without a runtime restart, so use the path directly.
sys.path.insert(0, str(REPO / "src"))
print("Repo:", REPO, "| Colab:", IN_COLAB)

# Pick up edits to src/solarcast/*.py without restarting. Colab's IPython 7 autoreload imports the
# removed `imp` module on Python 3.12+, so fall back gracefully instead of failing the whole notebook.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as exc:
    print(f"autoreload unavailable ({type(exc).__name__}); after editing src/, use Runtime > Restart session.")

## 1. Data and quality report (Site A)
Two-Site Solar PV Inverter & Weather Dataset (Romania), hourly. Missing measurements stay `NaN`; they are never filled with zeros.

In [ ]:
import pandas as pd

from solarcast import config, data

df_a = data.load_site("A")
print(pd.Series(data.data_quality_report(df_a)))
df_a.head()

## 2. Features and the no-leakage rule
Row `t` covers the hour ending at `t+1h`. At **issue time** (hour `t` is complete) we predict the power of hour `t+1`.
Inputs may only use rows up to `t`, plus the sun position and calendar of the target hour (known in advance).
Power or weather from `t+1` or later is never used (`tests/test_features.py` checks this).

- **Model A (power + solar):** recent power lags and rolling means, power at the same hour yesterday, sun position and time of day.
- **Model B (weather-aware):** Model A's features plus weather at issue time (cloud cover, irradiance, temperature, ...) and how fast cloud cover and irradiance are changing.
- **Baselines:** *Persistence* (next hour = this hour) and *Previous-day same hour*.
- **Splits (by time, never shuffled):** train 2025, validation Jan-Feb 2026, test Mar-Jun 2026. Models train on `train` only.

In [ ]:
from solarcast.features import FEATURE_GROUPS, build_features

for group, cols in FEATURE_GROUPS.items():
    print(f"{group:11s}", cols)
feat_a = build_features(df_a)
print(feat_a.shape)

## 3. Train and evaluate (Site A and Site B)

In [ ]:
from solarcast.pipeline import run_experiment

res_a = run_experiment("A")
res_b = run_experiment("B")

### Metrics: validation and test, Site A (daytime hours)
`nmae_pct` is MAE as a percentage of the 50 kW capacity.

In [ ]:
print("Validation (used for choices)")
display(res_a.metrics["val"].round(3))
print("Test (reported once)")
display(res_a.metrics["test"].round(3))

### Actual vs predicted, one sample test week

In [ ]:
from solarcast.evaluate import plot_actual_vs_predicted

plot_actual_vs_predicted(res_a.splits["test"], res_a.predictions["test"],
                        start="2026-05-18", end="2026-05-24")

## 4. Error analysis (test split, Site A)
Where does the model do well, and where does it fail?

In [ ]:
from solarcast.evaluate import grouped_metrics, plot_error_by_group

by_sky = grouped_metrics(res_a.splits["test"], res_a.predictions["test"], by="sky")
display(by_sky.round(3))
plot_error_by_group(by_sky)

In [ ]:
by_power = grouped_metrics(res_a.splits["test"], res_a.predictions["test"], by="power_bin")
display(by_power.round(3))
plot_error_by_group(by_power)

### Worst errors (Model B)

In [ ]:
from solarcast.evaluate import worst_errors

worst_errors(res_a.splits["test"], res_a.predictions["test"], "Model B", k=10).round(2)

## 5. What does Model B use? (permutation importance)
How much the test MAE gets worse when one feature is shuffled.

In [ ]:
from solarcast.models import permutation_importance_table

imp = permutation_importance_table(res_a.models["Model B"], res_a.splits["test"])
display(imp.head(12).round(4))
imp.head(12).set_index("feature")["importance_mean"].iloc[::-1].plot.barh(
    figsize=(7, 4), title="Permutation importance (MAE increase, kW)");

## 6. Cross-site check
Site A's trained models scored on Site B's test split (no retraining).

In [ ]:
from solarcast.pipeline import cross_site

print("Site B, own models")
display(res_b.metrics["test"].round(3))
print("Site B test split, Site A models")
display(cross_site(res_a, res_b).round(3))

## 7. The decision: when should the workshop charge its EV?
Our user runs a small workshop with this 50 kW rooftop system and one EV that needs 40 kWh every day
(11 kW charger, plugged in 08:00-18:00, workshop load 6 kW). These are **synthetic assumptions**; the solar
power is real. Each hour one fixed rule decides how much to charge:
`charge = clip(max(must_now, forecast - load), 0, min(11 kW, still needed))`, and the cost is judged with the
**actual** solar power of that hour. Only the forecast changes between strategies. Nothing here was tuned on test.

In [ ]:
from solarcast import charging

COLS = ["days", "ev_grid_kwh", "solar_share_pct", "grid_kwh_saved", "money_saved_eur", "co2_saved_kg", "pct_of_perfect"]
for site, res in [("A", res_a), ("B", res_b)]:
    preds = {k: v for k, v in res.predictions["test"].items() if k != "Previous-day same hour"}
    _, summary = charging.simulate(res.splits["test"], preds)
    print(f"Site {site}, test period (Mar-Jun 2026)")
    display(summary[COLS])

**What we found.** Any forecast-based plan buys far less grid power for the EV than charging on arrival.
But in spring and summer the simple persistence forecast makes *better* charging decisions than Model B, even though
Model B's forecasts are about twice as accurate. The cell below shows why: persistence lags behind the sunrise, so in the
morning it under-forecasts by 8-10 kW and waits. In summer waiting is free, because plenty of sun comes later. Model B is
unbiased, so on about half of the marginal morning hours it over-forecasts and the EV pulls grid power.

We tried a safety margin (charge only `forecast - load - margin`), choosing the margin on the **validation** period only
(`experiments/charging_margin.py`). In winter validation, waiting is expensive (little sun later), so the best margin was
0 and the result does not transfer to summer. **Lesson: a more accurate forecast is not automatically a more useful one.**
The rule also needs to know how much sun is still coming later in the day, which points to a multi-hour forecast as the
next step.

In [ ]:
slots = charging.daily_slots(res_a.splits["test"])
bias = pd.DataFrame({name: res_a.predictions["test"][name].reindex(slots.index) - slots["actual_kw"]
                     for name in ("Persistence", "Model B")})
bias["hour (local)"] = slots["hour"]
print("Mean forecast bias (forecast - actual, kW) by hour, Site A test period")
display(bias.groupby("hour (local)").mean().round(2))

## 8. Interactive demo
Pick a site and a daytime hour from the test split to see the one-hour-ahead forecast next to the baselines and the
actual value. In Colab this launches with a public `share=True` link.
*Historical replay on held-out test data. Prototype.*

In [ ]:
import os

from solarcast.demo import build_demo

demo = build_demo({"A": res_a, "B": res_b})
# Set SOLARCAST_LAUNCH_DEMO=0 to build the app without launching it (used for automated checks).
if os.environ.get("SOLARCAST_LAUNCH_DEMO", "1") != "0":
    demo.launch(share=IN_COLAB)  # Colab: public link; locally: inline

## Credits
- Dataset: *Two-Site Solar PV Inverter & Weather Dataset (Romania)*, https://github.com/upb-nlp/PV-Generation-Dataset, licensed CC BY 4.0. It includes Open-Meteo weather data.
- Solar position: pvlib (computed in the dataset).
- Libraries: scikit-learn, pandas, numpy, matplotlib, Gradio.
- AI tools: Claude Code was used to help write code and documentation.

This project is a prototype. Results are a historical replay; weather inputs are Open-Meteo reanalysis, not live forecasts.